# 第 27 章：光度红移估计案例

这个 notebook 把前面的回归、评估和可信度分析组合成一个更接近真实项目的小案例：

- 从一个 SDSS 风格的教学数据表开始
- 先做单色 baseline
- 再做多特征线性回归
- 比较测试集残差，并检查少见对象上的失效边界


In [ ]:
from __future__ import annotations

import csv
import math
from pathlib import Path

DATA_PATH = Path("../../data/small/photoz_case_demo.csv").resolve()

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            "galaxy_id": raw["galaxy_id"],
            "u_g": float(raw["u_g"]),
            "g_r": float(raw["g_r"]),
            "r_i": float(raw["r_i"]),
            "i_z": float(raw["i_z"]),
            "r_mag": float(raw["r_mag"]),
            "galaxy_type": raw["galaxy_type"],
            "specz": float(raw["specz"]),
        })

print(f"Loaded {len(rows)} galaxy samples from {DATA_PATH.name}")
print("specz range:", (min(row["specz"] for row in rows), max(row["specz"] for row in rows)))
type_counts = {}
for row in rows:
    type_counts[row["galaxy_type"]] = type_counts.get(row["galaxy_type"], 0) + 1
print("galaxy type counts:", type_counts)


In [ ]:
test_ids = {"Z003", "Z007", "Z012", "Z014", "Z017", "Z020"}
train_rows = [row for row in rows if row["galaxy_id"] not in test_ids]
test_rows = [row for row in rows if row["galaxy_id"] in test_ids]

def count_by_type(sample_rows):
    counts = {}
    for row in sample_rows:
        counts[row["galaxy_type"]] = counts.get(row["galaxy_type"], 0) + 1
    return counts

print("train size:", len(train_rows), "test size:", len(test_rows))
print("train type counts:", count_by_type(train_rows))
print("test ids:", [row["galaxy_id"] for row in test_rows])


In [ ]:
def fit_line(xs, ys):
    x_mean = sum(xs) / len(xs)
    y_mean = sum(ys) / len(ys)
    numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
    denominator = sum((x - x_mean) ** 2 for x in xs)
    slope = numerator / denominator
    intercept = y_mean - slope * x_mean
    return slope, intercept

def mean_absolute_error(y_true, y_pred):
    return sum(abs(a - b) for a, b in zip(y_true, y_pred)) / len(y_true)

def root_mean_square_error(y_true, y_pred):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(y_true, y_pred)) / len(y_true))

def mean_bias(y_true, y_pred):
    return sum(b - a for a, b in zip(y_true, y_pred)) / len(y_true)

baseline_slope, baseline_intercept = fit_line(
    [row["g_r"] for row in train_rows],
    [row["specz"] for row in train_rows],
)

baseline_predictions = []
for row in test_rows:
    prediction = baseline_intercept + baseline_slope * row["g_r"]
    baseline_predictions.append({
        "galaxy_id": row["galaxy_id"],
        "galaxy_type": row["galaxy_type"],
        "specz": row["specz"],
        "predicted_specz": prediction,
        "residual": prediction - row["specz"],
    })

print("baseline slope/intercept:", round(baseline_slope, 3), round(baseline_intercept, 3))
print(
    "baseline metrics:",
    {
        "mae": round(mean_absolute_error([row["specz"] for row in test_rows], [row["predicted_specz"] for row in baseline_predictions]), 4),
        "rmse": round(root_mean_square_error([row["specz"] for row in test_rows], [row["predicted_specz"] for row in baseline_predictions]), 4),
        "bias": round(mean_bias([row["specz"] for row in test_rows], [row["predicted_specz"] for row in baseline_predictions]), 4),
    },
)
for row in baseline_predictions:
    print(row["galaxy_id"], row["galaxy_type"], "true=", row["specz"], "pred=", round(row["predicted_specz"], 3), "residual=", round(row["residual"], 3))


In [ ]:
feature_names = ["u_g", "g_r", "r_i", "i_z", "r_mag"]

def fit_linear_model(sample_rows, feature_names, target_name):
    design_matrix = []
    targets = []
    for row in sample_rows:
        design_matrix.append([1.0] + [row[feature_name] for feature_name in feature_names])
        targets.append(row[target_name])

    matrix_size = len(design_matrix[0])
    xtx = [
        [sum(vector[i] * vector[j] for vector in design_matrix) for j in range(matrix_size)]
        for i in range(matrix_size)
    ]
    xty = [sum(vector[i] * target for vector, target in zip(design_matrix, targets)) for i in range(matrix_size)]

    augmented = [row[:] + [target] for row, target in zip(xtx, xty)]

    for column_index in range(matrix_size):
        pivot_index = max(range(column_index, matrix_size), key=lambda row_index: abs(augmented[row_index][column_index]))
        augmented[column_index], augmented[pivot_index] = augmented[pivot_index], augmented[column_index]
        pivot_value = augmented[column_index][column_index]
        for value_index in range(column_index, matrix_size + 1):
            augmented[column_index][value_index] /= pivot_value
        for row_index in range(matrix_size):
            if row_index == column_index:
                continue
            factor = augmented[row_index][column_index]
            for value_index in range(column_index, matrix_size + 1):
                augmented[row_index][value_index] -= factor * augmented[column_index][value_index]

    return [augmented[row_index][matrix_size] for row_index in range(matrix_size)]

def predict_linear_model(coefficients, row, feature_names):
    values = [1.0] + [row[feature_name] for feature_name in feature_names]
    return sum(weight * value for weight, value in zip(coefficients, values))

coefficients = fit_linear_model(train_rows, feature_names, "specz")
multifeature_predictions = []
for row in test_rows:
    prediction = predict_linear_model(coefficients, row, feature_names)
    multifeature_predictions.append({
        "galaxy_id": row["galaxy_id"],
        "galaxy_type": row["galaxy_type"],
        "specz": row["specz"],
        "predicted_specz": prediction,
        "residual": prediction - row["specz"],
    })

coefficient_summary = {"intercept": round(coefficients[0], 4)}
for feature_name, coefficient in zip(feature_names, coefficients[1:]):
    coefficient_summary[feature_name] = round(coefficient, 4)
print("multifeature coefficients:", coefficient_summary)
multi_true = [row["specz"] for row in test_rows]
multi_pred = [row["predicted_specz"] for row in multifeature_predictions]
baseline_true = [row["specz"] for row in test_rows]
baseline_pred = [row["predicted_specz"] for row in baseline_predictions]
print(
    "multifeature metrics:",
    {
        "mae": round(mean_absolute_error(multi_true, multi_pred), 4),
        "rmse": round(root_mean_square_error(multi_true, multi_pred), 4),
        "bias": round(mean_bias(multi_true, multi_pred), 4),
    },
)
print("test MAE improvement over baseline:", round(mean_absolute_error(baseline_true, baseline_pred) - mean_absolute_error(multi_true, multi_pred), 4))
for row in multifeature_predictions:
    print(row["galaxy_id"], row["galaxy_type"], "true=", row["specz"], "pred=", round(row["predicted_specz"], 3), "residual=", round(row["residual"], 3))


In [ ]:
baseline_by_id = {row["galaxy_id"]: row for row in baseline_predictions}
multifeature_by_id = {row["galaxy_id"]: row for row in multifeature_predictions}
residual_rows = []
for galaxy_id in sorted(multifeature_by_id):
    baseline_row = baseline_by_id[galaxy_id]
    multifeature_row = multifeature_by_id[galaxy_id]
    residual_rows.append({
        "galaxy_id": galaxy_id,
        "galaxy_type": multifeature_row["galaxy_type"],
        "specz": multifeature_row["specz"],
        "baseline_residual": baseline_row["residual"],
        "multifeature_residual": multifeature_row["residual"],
    })

residual_rows.sort(key=lambda row: abs(row["multifeature_residual"]), reverse=True)
print("Largest multifeature residuals:")
for row in residual_rows:
    print(
        row["galaxy_id"],
        row["galaxy_type"],
        "baseline=", round(row["baseline_residual"], 3),
        "multifeature=", round(row["multifeature_residual"], 3),
    )

mean_abs_residual_by_type = {}
for row in residual_rows:
    galaxy_type = row["galaxy_type"]
    mean_abs_residual_by_type.setdefault(galaxy_type, []).append(abs(row["multifeature_residual"]))

print("mean absolute residual by galaxy type:")
for galaxy_type, residual_values in mean_abs_residual_by_type.items():
    print(galaxy_type, round(sum(residual_values) / len(residual_values), 4))


In [ ]:
feature_means = {}
feature_stds = {}
for feature_name in feature_names:
    values = [row[feature_name] for row in train_rows]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    feature_means[feature_name] = mean_value
    feature_stds[feature_name] = math.sqrt(variance) or 1.0

def standardized_vector(row):
    return [
        (row[feature_name] - feature_means[feature_name]) / feature_stds[feature_name]
        for feature_name in feature_names
    ]

def euclidean_distance(left_vector, right_vector):
    return math.sqrt(sum((left - right) ** 2 for left, right in zip(left_vector, right_vector)))

print("Nearest training neighbors for test galaxies:")
for row in test_rows:
    vector = standardized_vector(row)
    neighbors = []
    for other_row in train_rows:
        other_vector = standardized_vector(other_row)
        neighbors.append((
            euclidean_distance(vector, other_vector),
            other_row["galaxy_id"],
            other_row["galaxy_type"],
        ))
    neighbors.sort(key=lambda item: item[0])
    nearest = [(round(distance, 3), galaxy_id, galaxy_type) for distance, galaxy_id, galaxy_type in neighbors[:3]]
    print(row["galaxy_id"], row["galaxy_type"], nearest)

print("Note: only one blue_compact galaxy appears in the training sample (Z019).")


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Optional matplotlib plot skipped: {exc}")
else:
    baseline_lookup = {row["galaxy_id"]: row["predicted_specz"] for row in baseline_predictions}
    multifeature_lookup = {row["galaxy_id"]: row["predicted_specz"] for row in multifeature_predictions}
    true_values = [row["specz"] for row in test_rows]
    baseline_values = [baseline_lookup[row["galaxy_id"]] for row in test_rows]
    multifeature_values = [multifeature_lookup[row["galaxy_id"]] for row in test_rows]
    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].scatter(true_values, baseline_values, label="baseline", s=50)
    axes[0].scatter(true_values, multifeature_values, label="multifeature", s=50)
    axes[0].plot([0.0, 0.6], [0.0, 0.6], linestyle="--", color="black", linewidth=1)
    axes[0].set_xlabel("true specz")
    axes[0].set_ylabel("predicted specz")
    axes[0].set_title("Test-set predictions")
    axes[0].legend()
    axes[1].axhline(0.0, linestyle="--", color="black", linewidth=1)
    axes[1].scatter(true_values, [value - truth for value, truth in zip(baseline_values, true_values)], label="baseline", s=50)
    axes[1].scatter(true_values, [value - truth for value, truth in zip(multifeature_values, true_values)], label="multifeature", s=50)
    axes[1].set_xlabel("true specz")
    axes[1].set_ylabel("residual")
    axes[1].set_title("Residual comparison")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


In [ ]:
try:
    from sklearn.ensemble import RandomForestRegressor
except Exception as exc:
    print(f"Optional scikit-learn comparison skipped: {exc}")
else:
    train_matrix = [[row[feature_name] for feature_name in feature_names] for row in train_rows]
    test_matrix = [[row[feature_name] for feature_name in feature_names] for row in test_rows]
    forest = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=7)
    forest.fit(train_matrix, [row["specz"] for row in train_rows])
    forest_predictions = forest.predict(test_matrix)
    print(
        "random forest metrics:",
        {
            "mae": round(mean_absolute_error([row["specz"] for row in test_rows], forest_predictions), 4),
            "rmse": round(root_mean_square_error([row["specz"] for row in test_rows], forest_predictions), 4),
            "bias": round(mean_bias([row["specz"] for row in test_rows], forest_predictions), 4),
        },
    )


这个案例最值得记住的地方，不是某个具体误差数字，而是下面这条经验：

> 整体指标提升，并不代表所有对象都变得同样可靠。

在真实光度红移项目里，我们同样需要回到对象类型、样本覆盖范围和训练分布本身，判断模型是否值得信任。 
